# 03 · Exploratory Data Analysis

**Stage 3 of 8** · ACCESS → WRANGLE → EXPLORE → VISUALIZE → ENGINEER → MODEL → EVALUATE → RECOMMEND

> **Project premise:** if someone has had cancer and was treated, **will it come back?** And for every factor linked to it coming back: **correlation or causation?**
>
> All data here is public: a k-anonymous feature table (every combination of site group, region, sex and diagnosis period is shared by ≥ 3 patients), synthetic patients, and aggregate reports. The raw database and clinical notes stay private.

**Goal of this stage:** understand what the cleaned data says before building any model.

**Business question:** How often does cancer come back, and which factors differ between patients whose cancer came back and those whose didn't?

In [1]:
%matplotlib inline
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
PUBLIC = ROOT / "data" / "public" / "patients_features_public.csv"
SYNTH = ROOT / "data" / "synthetic" / "synthetic_patients.csv"
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8984"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.axisbelow": True, "figure.dpi": 110})
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
df = pd.read_csv(PUBLIC)

In [2]:
at_risk = df[df.eligible_for_relapse == 1].copy()
at_risk["relapse"] = at_risk.relapse.astype(int)

### Q1 · Descriptive statistics for the key numeric variables

In [3]:
df[["years_since_dx", "rfs_years", "n_met_sites", "n_visits"]].describe().round(2)

,years_since_dx,rfs_years,n_met_sites,n_visits
count,250.00,138.00,335.00,335.00
mean,2.83,4.07,0.36,3.73
std,3.04,2.77,0.60,2.98
min,0.00,1.00,0.00,0.00
25%,0.00,1.00,0.00,1.00
50%,1.00,4.00,0.00,3.00
75%,5.75,6.00,1.00,5.00
max,10.00,10.00,3.00,20.00


### Q2 · How many relapsed vs did not? (the headline KPI)

In [4]:
at_risk.relapse.value_counts().rename({0: "did not come back", 1: "came back"}).to_frame("patients")

,patients
relapse,
did not come back,108
came back,30


## KPIs
| Clinic goal | KPI | Formula |
|---|---|---|
| Know how often cancer comes back | **Relapse rate** | relapses ÷ at-risk patients |
| Earlier diagnosis | **% metastatic at diagnosis** | metastatic at dx ÷ all patients |
| Better records | **% breast patients with receptor status** | breast with known subtype ÷ breast |
| Palliative capacity | **% referred to palliative care** | referred ÷ all patients |

In [5]:
breast = df[df.site_group == "Breast"]
kpi = pd.Series({
    "Relapse rate (at-risk)": at_risk.relapse.mean(),
    "% metastatic at diagnosis": df.metastatic_at_dx.mean(),
    "% breast with receptor status": (breast.breast_subtype.fillna("Unknown") != "Unknown").mean(),
    "% referred to palliative care": df.palliative_referral.mean(),
}).map("{:.0%}".format)
kpi.to_frame("value")

,value
Relapse rate (at-risk),22%
% metastatic at diagnosis,19%
% breast with receptor status,32%
% referred to palliative care,17%


### Q3 · Does the relapse rate differ by cancer type?

In [6]:
by_site = at_risk.groupby("site_group").relapse.agg(patients="size", relapses="sum", rate="mean")
by_site.sort_values("patients", ascending=False).style.format({"rate": "{:.0%}"})

,patients,relapses,rate
site_group,,,
Breast,94,18,19%
Head & neck,9,4,44%
Colorectal / anal,7,2,29%
CNS,6,0,0%
Genitourinary,5,2,40%
Gynaecologic,5,1,20%
Skin,4,0,0%
Sarcoma,3,2,67%
Upper GI / hepatobiliary,3,1,33%


### Q4 · Does it differ by treatment received?

In [7]:
rows = {}
for t in ["surgery", "radiotherapy", "hormonal_therapy", "neoadjuvant_chemo", "treated_abroad"]:
    g = at_risk.groupby(t).relapse.mean()
    rows[t] = {"rate if yes": g.get(1), "rate if no": g.get(0), "n yes": int((at_risk[t] == 1).sum())}
pd.DataFrame(rows).T.style.format({"rate if yes": "{:.0%}", "rate if no": "{:.0%}"})

,rate if yes,rate if no,n yes
surgery,23%,21%,43.000000
radiotherapy,18%,27%,83.000000
hormonal_therapy,19%,25%,70.000000
neoadjuvant_chemo,23%,22%,13.000000
treated_abroad,30%,21%,10.000000


### Q5 · Does follow-up time differ? (why time matters)

In [8]:
at_risk.groupby("relapse").rfs_years.describe().round(1)

,count,mean,std,min,25%,50%,75%,max
relapse,,,,,,,,
0,108.0,4.2,2.8,1.0,1.0,4.5,6.0,10.0
1,30.0,3.7,2.5,1.0,2.0,2.5,5.8,10.0


Patients who did **not** relapse have often been followed only a short time. A yes/no rate treats
"not back after 1 year" the same as "not back after 10". That's why the later notebooks use
**time-to-event** methods (Kaplan–Meier, Cox).

### Q6 · Relationship between two numeric variables: follow-up length vs number of visits

In [9]:
print("correlation (Spearman):", round(at_risk[["years_since_dx", "n_visits"]].corr(method="spearman").iloc[0, 1], 2))

correlation (Spearman): -0.01


### Q7 · Which factors move together? (correlation is not causation)

In [10]:
cols = ["surgery", "radiotherapy", "hormonal_therapy", "neoadjuvant_chemo", "relapse"]
X = at_risk[cols].astype(float).assign(breast=(at_risk.site_group == "Breast").astype(float))
X.corr().round(2)

,surgery,radiotherapy,hormonal_therapy,neoadjuvant_chemo,relapse,breast
surgery,1.00,0.16,0.13,0.10,0.02,0.09
radiotherapy,0.16,1.00,0.06,0.11,-0.11,-0.05
hormonal_therapy,0.13,0.06,1.00,0.17,-0.08,0.66
neoadjuvant_chemo,0.10,0.11,0.17,1.00,0.01,0.17
relapse,0.02,-0.11,-0.08,0.01,1.00,-0.09
breast,0.09,-0.05,0.66,0.17,-0.09,1.00


**Two patterns worth investigating:**
1. **Breast cancers came back less often** than other cancers.
2. **Hormonal therapy and radiotherapy go with less relapse**, but hormonal therapy is strongly tied to having
   breast cancer, so part of that link may simply be "breast cancer". EDA can't separate the two;
   notebook 08 tries to.

## So what?

About **1 in 5 treated patients (22%)** saw their cancer come back. Breast cancer and some treatments go with
less relapse, but these are **correlations**, and the factors are tangled together (hormonal therapy ≈ breast).
Follow-up length varies a lot, so the next notebooks use time-aware methods.